In [45]:
import os

# 1. Install CrewAI
!pip install -q crewai

# 2. Create the flattened directory structure
PROJECT_NAME = "blog_writing_project"
folders = [
    f"{PROJECT_NAME}/config",
    f"{PROJECT_NAME}/blogs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

# 3. Add an empty __init__.py to make it a package
with open(f"{PROJECT_NAME}/__init__.py", "w") as f:
    pass

print(f"Project structure for '{PROJECT_NAME}' created successfully.")

Project structure for 'blog_writing_project' created successfully.


In [46]:
%%writefile blog_writing_project/config/agents.yaml
report_generator:
  role: >
    Expert report generator on {topic}
  goal: >
    Creating a detailed report on {topic}
  backstory: >
    You are an expert researcher with past experience creating
    detailed reports covering all dimensions of a topic.

blog_writer:
  role: >
    Expert blog writer on a given topic.
  goal: >
    Creating a well-crafted blog on {topic}
  backstory: >
    You are an expert in creating engaging blogs that are simple
    and fun to read.

Overwriting blog_writing_project/config/agents.yaml


In [47]:
%%writefile blog_writing_project/config/tasks.yaml
report_task:
  description: >
    Based on the input topic: {topic}, generate a detailed report
    including all the latest facts and future trends.
  expected_output: >
    2000 words long report on topic: {topic}.
  agent: >
    report_generator

blog_writing_task:
  description: >
    Based on the report on {topic}, create a well-written and
    structured blog that even a 5-year-old can understand.
  expected_output: >
    500 words long blog post on topic: {topic}.
  agent: >
    blog_writer

Overwriting blog_writing_project/config/tasks.yaml


In [48]:
import yaml

# Defining the dictionaries directly in Python to avoid string-parsing issues
agents_data = {
    "report_generator": {
        "role": "Expert report generator on {topic}",
        "goal": "Creating a detailed report on {topic}",
        "backstory": "You are an expert researcher with past experience creating detailed reports covering all dimensions of a topic."
    },
    "blog_writer": {
        "role": "Expert blog writer on a given topic.",
        "goal": "Creating a well-crafted blog on {topic}",
        "backstory": "You are an expert in creating engaging blogs that are simple and fun to read."
    }
}

tasks_data = {
    "report_task": {
        "description": "Based on the input topic: {topic}, generate a detailed report including all the latest facts and future trends.",
        "expected_output": "2000 words long report on topic: {topic}.",
        "agent": "report_generator"
    },
    "blog_writing_task": {
        "description": "Based on the report on {topic}, create a well-written and structured blog that even a 5-year-old can understand.",
        "expected_output": "500 words long blog post on topic: {topic}.",
        "agent": "blog_writer"
    }
}

# Safely write them to YAML
with open('blog_writing_project/config/agents.yaml', 'w') as f:
    yaml.dump(agents_data, f)

with open('blog_writing_project/config/tasks.yaml', 'w') as f:
    yaml.dump(tasks_data, f)

print("✅ Configuration files cleaned and updated!")

✅ Configuration files cleaned and updated!


In [49]:
%%writefile blog_writing_project/crew.py
from crewai import Agent, Crew, Process, Task
from crewai import CrewBase, agent, crew, task
import os

@CrewBase
class BlogWritingWithCrewAI:
    # Dynamically locate the config folder relative to this file
    base_path = os.path.dirname(__file__)
    agents_config = os.path.join(base_path, 'config/agents.yaml')
    tasks_config = os.path.join(base_path, 'config/tasks.yaml')

    @agent
    def report_generator(self) -> Agent:
        return Agent(config=self.agents_config['report_generator'])

    @agent
    def blog_writer(self) -> Agent:
        return Agent(config=self.agents_config['blog_writer'])

    @task
    def report_task(self) -> Task:
        return Task(config=self.tasks_config['report_task'])

    @task
    def blog_writing_task(self) -> Task:
        return Task(
            config=self.tasks_config['blog_writing_task'],
            output_file=os.path.join(self.base_path, '/content/blog_writing_project/blogs/blog.md')
        )

    @crew
    def crew(self) -> Crew:
        return Crew(
            agents=self.agents,
            tasks=self.tasks,
            process=Process.sequential,
            verbose=True
        )

Overwriting blog_writing_project/crew.py


In [50]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [51]:
import crewai
import crewai.project
print(f"CrewAI Version: {crewai.__version__}")
# If this doesn't error, you are ready to run.

CrewAI Version: 1.11.0


In [52]:
%%writefile blog_writing_project/main.py
from blog_writing_project.crew import BlogWritingWithCrewAI

def run(topic_name): # We now pass the topic here
    inputs = {
        "topic": topic_name
    }
    try:
        # Re-initialize the crew with the fresh inputs
        result = BlogWritingWithCrewAI().crew().kickoff(inputs=inputs)
        return result
    except Exception as e:
        print(f"An error occurred: {e}")

Overwriting blog_writing_project/main.py


In [53]:
import sys
import os
import importlib

# 1. Ensure Pathing
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 2. Force Reload (Clears the 'Vibe Coding' memory)
import blog_writing_project.main
importlib.reload(blog_writing_project.main)
from blog_writing_project.main import run

# 3. SET YOUR TOPIC HERE
current_topic = "Multi-agent systems"

print(f"🚀 Starting CrewAI for: {current_topic}...")
output = run(current_topic)

print("\n--- Final Output ---\n")
print(output)

🚀 Starting CrewAI for: Multi-agent systems...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 36e9f03d-ab73-4c9c-806e-68a5532e1ac7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: report_task                                                                                              │
│  ID: 05a65377-b0f4-4cbf-b278-bda7ba9685fc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert report generator on Multi-agent systems                                                          │
│                                                                                                                 │
│  Task: Based on the input topic: Multi-agent systems, generate a detailed report including all the latest       │
│  facts and future trends.                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert report generator on Multi-agent systems                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Detailed Report on Multi-Agent Systems                                                                       │
│                                                                                                                 │
│  ## Table of Contents                                                                                           │
│                                                                                                                 │
│  1. Introduction                                                                                                │
│  2. Definitions and Core Concepts                                                                               │
│  3. Classification and Architectures of Multi-Agent Systems                                                     │
│  4. Key Technologies and Components                                                                             │
│  5. Application Areas                                                                                           │
│  6. Current Trends and Recent Advances                                                                          │
│  7. Challenges and Limitations                                                                                  │
│  8. Future Prospects and Emerging Directions                                                                    │
│  9. Conclusion                                                                                                  │
│  10. References                                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Introduction                                                                                             │
│                                                                                                                 │
│  Multi-agent systems (MAS) represent an advanced subfield of distributed artificial intelligence (DAI) and      │
│  computer science, focused on the design and implementation of systems composed of multiple interacting         │
│  intelligent agents. These agents are autonomous entities capable of perceiving their environments, making      │
│  decisions, and acting to achieve specific goals, often working collaboratively or competitively within         │
│  complex and dynamic environments.                                                                              │
│                                                                                                                 │
│  The field of MAS has gained significant attention over the last two decades due to its applicability in        │
│  solving large-scale, complex problems that are not easily tackled by monolithic systems. With the              │
│  proliferation of interconnected devices, the Internet of Things (IoT), and the pressing need for               │
│  decentralized autonomous operations, MAS has become a cornerstone technology across numerous domains           │
│  including robotics, economics, telecommunications, cyb

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: blog_writing_task                                                                                        │
│  ID: 906d94e4-8f42-44df-a73c-ba8ee140c435                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert blog writer on a given topic.                                                                    │
│                                                                                                                 │
│  Task: Based on the report on Multi-agent systems, create a well-written and structured blog that even a        │
│  5-year-old can understand.                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert blog writer on a given topic.                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Multi-Agent Systems: A Simple and Fun Guide for Everyone!**                                                  │
│                                                                                                                 │
│  Have you ever wondered how lots of little robots or computers can work together like a team to do cool         │
│  things? Imagine a group of smart helpers, each with their own superpowers, talking and cooperating to get big  │
│  jobs done. That’s what a **Multi-Agent System** is all about! Let’s take a fun journey to understand this      │
│  amazing idea in a way even a 5-year-old can enjoy!                                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### What is a Multi-Agent System?                                                                              │
│                                                                                                                 │
│  Think about ants crawling around, each doing a different job but all helping their home. Ants don’t have a     │
│  boss telling them what to do, but somehow they work together perfectly. Multi-agent systems are like that!     │
│                                                                                                                 │
│  - **Agents** are the smart little helpers — they can be robots, software programs, or anything that can        │
│  sense, think a bit, and act.                                                                                   │
│  - When many agents work together inside a system, we call it a **multi-agent system**.                         │
│                                                                                                                 │
│  Just like ants, these agents can see what’s happening around them, talk to each other, and try to reach goals  │
│  — like finding food or building a nest.                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### What Can These Agents Do?                                                                                  │
│                                                                                                                 │
│  Every agent is like a tiny superhero with special powers:                                                      │
│                                                                                                                 │
│  1. **Seeing:** Agents notice what’s happening (like eyes and ears).                                            │
│  2. **Thinking:** They decide what to do next (like a brain).                                                   │
│  3. **Doing:** They take actions (like hands or wheels)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: blog_writing_task                                                                                        │
│  Agent: Expert blog writer on a given topic.                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Final Output ---

**Multi-Agent Systems: A Simple and Fun Guide for Everyone!**

Have you ever wondered how lots of little robots or computers can work together like a team to do cool things? Imagine a group of smart helpers, each with their own superpowers, talking and cooperating to get big jobs done. That’s what a **Multi-Agent System** is all about! Let’s take a fun journey to understand this amazing idea in a way even a 5-year-old can enjoy!

---

### What is a Multi-Agent System?

Think about ants crawling around, each doing a different job but all helping their home. Ants don’t have a boss telling them what to do, but somehow they work together perfectly. Multi-agent systems are like that! 

- **Agents** are the smart little helpers — they can be robots, software programs, or anything that can sense, think a bit, and act.
- When many agents work together inside a system, we call it a **multi-agent system**.

Just like ants, these agents can see what’s happening around them,

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 36e9f03d-ab73-4c9c-806e-68a5532e1ac7                                                                       │
│  Final Output: **Multi-Agent Systems: A Simple and Fun Guide for Everyone!**                                    │
│                                                                                                                 │
│  Have you ever wondered how lots of little robots or computers can work together like a team to do cool         │
│  things? Imagine a group of smart helpers, each with their own superpowers, talking and cooperating to get big  │
│  jobs done. That’s what a **Multi-Agent System** is all about! Let’s take a fun journey to understand this      │
│  amazing idea in a way even a 5-year-old can enjoy!                                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### What is a Multi-Agent System?                                                                              │
│                                                                                                                 │
│  Think about ants crawling around, each doing a different job but all helping their home. Ants don’t have a     │
│  boss telling them what to do, but somehow they work together perfectly. Multi-agent systems are like that!     │
│                                                                                                                 │
│  - **Agents** are the smart little helpers — they can be robots, software programs, or anything that can        │
│  sense, think a bit, and act.                                                                                   │
│  - When many agents work together inside a system, we call it a **multi-agent system**.                         │
│                                                                                                                 │
│  Just like ants, these agents can see what’s happening around them, talk to each other, and try to reach goals  │
│  — like finding food or building a nest.                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### What Can These Agents Do?                                                                                  │
│                                                                                                                 │
│  Every agent is like a tiny superhero with special powers:                                                      │
│                                                                                                                 │
│  1. **Seeing:** Agents notice what’s happening (like eyes and ears).                                            │
│  2. **Thinking:** They decide what to do next (like a brain).                                                   │
│  3. **Doing:** They take actions (like hands or wheels

In [54]:
!zip -r /content/blog_writing_project.zip /content/blog_writing_project/

  adding: content/blog_writing_project/ (stored 0%)
  adding: content/blog_writing_project/crew.py (deflated 64%)
  adding: content/blog_writing_project/__pycache__/ (stored 0%)
  adding: content/blog_writing_project/__pycache__/crew.cpython-312.pyc (deflated 54%)
  adding: content/blog_writing_project/__pycache__/main.cpython-312.pyc (deflated 25%)
  adding: content/blog_writing_project/__pycache__/__init__.cpython-312.pyc (deflated 22%)
  adding: content/blog_writing_project/main.py (deflated 41%)
  adding: content/blog_writing_project/blogs/ (stored 0%)
  adding: content/blog_writing_project/config/ (stored 0%)
  adding: content/blog_writing_project/config/agents.yaml (deflated 50%)
  adding: content/blog_writing_project/config/tasks.yaml (deflated 49%)
  adding: content/blog_writing_project/__init__.py (stored 0%)
